# DIAGNOSTIC


In [2]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Verify the structure
BASE = '/content/drive/MyDrive/ECG project Amirhossein taleshi'
print(f'\n📁 {BASE}')
for f in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, f)
    if os.path.isdir(full):
        n = sum(1 for _ in os.scandir(full))
        print(f'  📁 {f}/  ({n} items)')
    else:
        print(f'  📄 {f}')

# Then drill into results
results = f'{BASE}/results'
if os.path.isdir(results):
    print(f'\n📁 {results}/')
    for f in sorted(os.listdir(results)):
        full = os.path.join(results, f)
        if os.path.isdir(full):
            print(f'  📁 {f}/')
            # one level deeper
            for g in sorted(os.listdir(full))[:5]:
                print(f'     {g}')
        else:
            print(f'  📄 {f}')

Mounted at /content/drive

📁 /content/drive/MyDrive/ECG project Amirhossein taleshi
  📁 LTAFDB_benchmark/  (41 items)
  📄 Report.docx
  📁 checkpoints/  (4 items)
  📁 fix code/  (5 items)
  📁 processed_datasets/  (6 items)
  📁 prof file/  (10 items)
  📁 results/  (2 items)

📁 /content/drive/MyDrive/ECG project Amirhossein taleshi/results/
  📄 AFDB_full_results.zip
  📁 FILES of BENCHMARK/
     AFDB_benchmark
     CPSC2021_benchmark
     CinC2017_benchmark
     LTAFDB_benchmark


In [3]:
import os

BASE = '/content/drive/MyDrive/ECG project Amirhossein taleshi'
EXPECTED = ['HuBERT-Small','HuBERT-Base','HuBERT-Large',
            'CLEF-Small','CLEF-Medium','CLEF-Large',
            'ST-MEM','ECG-JEPA','ECGFounder']
SUFFIXES = ['_results.csv','_predictions.csv','_efficiency.csv','_embs.npy']

# Canonical paths (use the NEW LTAFDB at top-level)
PATHS = {
    'CinC2017': f'{BASE}/results/FILES of BENCHMARK/CinC2017_benchmark',
    'AFDB'    : f'{BASE}/results/FILES of BENCHMARK/AFDB_benchmark',
    'LTAFDB'  : f'{BASE}/LTAFDB_benchmark',                          # ← NEW one
    'CPSC2021': f'{BASE}/results/FILES of BENCHMARK/CPSC2021_benchmark',
}

# Optional: rename the old LTAFDB so we don't confuse it
old = f'{BASE}/results/FILES of BENCHMARK/LTAFDB_benchmark'
if os.path.isdir(old) and not os.path.isdir(old + '_OLD'):
    os.rename(old, old + '_OLD')
    print(f'📁 Renamed old folder → LTAFDB_benchmark_OLD\n')

print('='*80)
print(' '*20 + 'PER-MODEL COVERAGE AUDIT (correct paths)')
print('='*80)

action = {ds:{'missing_npy':[],'missing_pred':[],'missing_results':[],'missing_eff':[]}
          for ds in PATHS}

for ds, folder in PATHS.items():
    print(f'\n📁 {ds}  →  {folder}')
    if not os.path.isdir(folder):
        print(f'   ❌ folder missing'); continue
    print(f'   {"Model":<15} {"results":>8} {"preds":>8} {"effic":>8} {"embs":>10}')
    print(f'   {"-"*15} {"-"*8} {"-"*8} {"-"*8} {"-"*10}')
    for m in EXPECTED:
        marks = []
        for suf in SUFFIXES:
            f = f'{folder}/{m}{suf}'
            if os.path.exists(f) and os.path.getsize(f) > 100:
                marks.append('✅')
            else:
                marks.append('❌')
                key = {'_results.csv':'missing_results',
                       '_predictions.csv':'missing_pred',
                       '_efficiency.csv':'missing_eff',
                       '_embs.npy':'missing_npy'}[suf]
                action[ds][key].append(m)
        print(f'   {m:<15} {marks[0]:>8} {marks[1]:>8} {marks[2]:>8} {marks[3]:>10}')

print('\n' + '='*80)
print('ACTION ITEMS')
print('='*80)
all_clean = True
for ds, items in action.items():
    if not any(items.values()):
        print(f'\n✅ {ds}: complete')
        continue
    all_clean = False
    print(f'\n📁 {ds}:')
    if items['missing_npy']:    print(f'   need .npy (re-embed):    {items["missing_npy"]}')
    if items['missing_pred']:   print(f'   need predictions (XGB):  {items["missing_pred"]}')
    if items['missing_results']:print(f'   need results.csv:        {items["missing_results"]}')
    if items['missing_eff']:    print(f'   need efficiency.csv:     {items["missing_eff"]}')

print('\n' + ('🎉 ALL DATASETS COMPLETE' if all_clean else '⚠ Some files missing — see above'))

📁 Renamed old folder → LTAFDB_benchmark_OLD

                    PER-MODEL COVERAGE AUDIT (correct paths)

📁 CinC2017  →  /content/drive/MyDrive/ECG project Amirhossein taleshi/results/FILES of BENCHMARK/CinC2017_benchmark
   Model            results    preds    effic       embs
   --------------- -------- -------- -------- ----------
   HuBERT-Small           ✅        ✅        ✅          ✅
   HuBERT-Base            ✅        ✅        ✅          ✅
   HuBERT-Large           ✅        ✅        ✅          ✅
   CLEF-Small             ✅        ✅        ✅          ✅
   CLEF-Medium            ✅        ✅        ✅          ✅
   CLEF-Large             ✅        ✅        ✅          ✅
   ST-MEM                 ✅        ✅        ✅          ✅
   ECG-JEPA               ✅        ✅        ✅          ✅
   ECGFounder             ✅        ✅        ✅          ✅

📁 AFDB  →  /content/drive/MyDrive/ECG project Amirhossein taleshi/results/FILES of BENCHMARK/AFDB_benchmark
   Model            results    preds    e

In [4]:
import os
BASE = '/content/drive/MyDrive/ECG project Amirhossein taleshi'

# List everything that has "LTAFDB" in its name
print('All LTAFDB-related folders/files:\n')
for root, dirs, files in os.walk(BASE):
    for d in dirs:
        if 'LTAFDB' in d or 'ltafdb' in d.lower():
            full = os.path.join(root, d)
            n = sum(1 for _ in os.scandir(full))
            sz = sum(os.path.getsize(os.path.join(full,f)) for f in os.listdir(full)
                     if os.path.isfile(os.path.join(full,f))) / 1e6
            print(f'  📁 {full}')
            print(f'     ({n} items, {sz:.0f} MB)')

All LTAFDB-related folders/files:

  📁 /content/drive/MyDrive/ECG project Amirhossein taleshi/prof file/ltafdb
     (336 items, 3650 MB)
  📁 /content/drive/MyDrive/ECG project Amirhossein taleshi/results/FILES of BENCHMARK/LTAFDB_benchmark(NEW AND CORRECT)
     (41 items, 46485 MB)
  📁 /content/drive/MyDrive/ECG project Amirhossein taleshi/results/FILES of BENCHMARK/LTAFDB_benchmark_OLD
     (40 items, 46548 MB)


In [5]:
import os

BASE = '/content/drive/MyDrive/ECG project Amirhossein taleshi'
EXPECTED = ['HuBERT-Small','HuBERT-Base','HuBERT-Large',
            'CLEF-Small','CLEF-Medium','CLEF-Large',
            'ST-MEM','ECG-JEPA','ECGFounder']
SUFFIXES = ['_results.csv','_predictions.csv','_efficiency.csv','_embs.npy']

PATHS = {
    'CinC2017': f'{BASE}/results/FILES of BENCHMARK/CinC2017_benchmark',
    'AFDB'    : f'{BASE}/results/FILES of BENCHMARK/AFDB_benchmark',
    'LTAFDB'  : f'{BASE}/results/FILES of BENCHMARK/LTAFDB_benchmark(NEW AND CORRECT)',
    'CPSC2021': f'{BASE}/results/FILES of BENCHMARK/CPSC2021_benchmark',
}

print('='*80)
print(' '*22 + 'PER-MODEL COVERAGE AUDIT')
print('='*80)

action = {ds:{'missing_npy':[],'missing_pred':[],'missing_results':[],'missing_eff':[]}
          for ds in PATHS}

for ds, folder in PATHS.items():
    print(f'\n📁 {ds}')
    if not os.path.isdir(folder):
        print(f'   ❌ folder missing: {folder}')
        # mark all 9 missing for everything
        for m in EXPECTED:
            for k in action[ds]: action[ds][k].append(m)
        continue
    print(f'   {"Model":<15} {"results":>8} {"preds":>8} {"effic":>8} {"embs":>10}')
    print(f'   {"-"*15} {"-"*8} {"-"*8} {"-"*8} {"-"*10}')
    for m in EXPECTED:
        marks = []
        for suf in SUFFIXES:
            f = f'{folder}/{m}{suf}'
            if os.path.exists(f) and os.path.getsize(f) > 100:
                marks.append('✅')
            else:
                marks.append('❌')
                key = {'_results.csv':'missing_results',
                       '_predictions.csv':'missing_pred',
                       '_efficiency.csv':'missing_eff',
                       '_embs.npy':'missing_npy'}[suf]
                action[ds][key].append(m)
        print(f'   {m:<15} {marks[0]:>8} {marks[1]:>8} {marks[2]:>8} {marks[3]:>10}')

print('\n' + '='*80 + '\nACTION ITEMS\n' + '='*80)
all_clean = True
for ds, items in action.items():
    if not any(items.values()):
        print(f'\n✅ {ds}: complete'); continue
    all_clean = False
    print(f'\n📁 {ds}:')
    if items['missing_npy']:    print(f'   need .npy:               {items["missing_npy"]}')
    if items['missing_pred']:   print(f'   need predictions.csv:    {items["missing_pred"]}')
    if items['missing_results']:print(f'   need results.csv:        {items["missing_results"]}')
    if items['missing_eff']:    print(f'   need efficiency.csv:     {items["missing_eff"]}')

print('\n' + ('🎉 ALL COMPLETE' if all_clean else '⚠ Work needed — see above'))

                      PER-MODEL COVERAGE AUDIT

📁 CinC2017
   Model            results    preds    effic       embs
   --------------- -------- -------- -------- ----------
   HuBERT-Small           ✅        ✅        ✅          ✅
   HuBERT-Base            ✅        ✅        ✅          ✅
   HuBERT-Large           ✅        ✅        ✅          ✅
   CLEF-Small             ✅        ✅        ✅          ✅
   CLEF-Medium            ✅        ✅        ✅          ✅
   CLEF-Large             ✅        ✅        ✅          ✅
   ST-MEM                 ✅        ✅        ✅          ✅
   ECG-JEPA               ✅        ✅        ✅          ✅
   ECGFounder             ✅        ✅        ✅          ✅

📁 AFDB
   Model            results    preds    effic       embs
   --------------- -------- -------- -------- ----------
   HuBERT-Small           ✅        ✅        ✅          ✅
   HuBERT-Base            ✅        ✅        ✅          ✅
   HuBERT-Large           ✅        ✅        ✅          ✅
   CLEF-Small        

# GENERATE THE REQUIERMENTS

In [6]:
# ============================================================
# Generate the 3 professor deliverables per dataset:
#   1. <DATASET>_all_predictions.csv
#   2. <DATASET>_efficiency_all.csv
#   3. <DATASET>_tsne_coords.csv
# ============================================================
import os, gc, numpy as np, pandas as pd
import pyarrow.parquet as pq
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

BASE = '/content/drive/MyDrive/ECG project Amirhossein taleshi'
EXPECTED = ['HuBERT-Small','HuBERT-Base','HuBERT-Large',
            'CLEF-Small','CLEF-Medium','CLEF-Large',
            'ST-MEM','ECG-JEPA','ECGFounder']
N_PER_CLASS = 1500

DATASETS = {
    'CinC2017': (f'{BASE}/results/FILES of BENCHMARK/CinC2017_benchmark',
                 'cinc2017_300hz_lead1_10s_5s.parquet'),
    'AFDB'    : (f'{BASE}/results/FILES of BENCHMARK/AFDB_benchmark',
                 'afdb_250hz_lead2_10s_5s.parquet'),
    'LTAFDB'  : (f'{BASE}/results/FILES of BENCHMARK/LTAFDB_benchmark(NEW AND CORRECT)',
                 'ltafdb_128hz_lead2_10s_5s.parquet'),
    'CPSC2021': (f'{BASE}/results/FILES of BENCHMARK/CPSC2021_benchmark',
                 'cpsc2021_200hz_lead2_10s_5s.parquet'),
}

for ds, (folder, parq) in DATASETS.items():
    PARQUET = f'{BASE}/processed_datasets/{parq}'
    print(f'\n{"="*60}\n  {ds}\n{"="*60}')

    # Load labels + record_name (no signals)
    cols = pq.read_table(PARQUET, columns=None).column_names
    rec_col = 'record_name' if 'record_name' in cols else 'record_id'
    t = pq.read_table(PARQUET, columns=['Labels', rec_col])
    lab  = t['Labels'].to_numpy().astype(np.int64)
    recs = t[rec_col].to_numpy()
    del t; gc.collect()

    # 1. all_predictions.csv
    base = pd.DataFrame({'window_idx':np.arange(len(lab)),
                         'record_name':recs, 'true_label':lab})
    merged = base.copy()
    for m in EXPECTED:
        f = f'{folder}/{m}_predictions.csv'
        if not os.path.exists(f): continue
        p = pd.read_csv(f)[['window_idx','probability','prediction','fold']].rename(
            columns={'probability':f'{m}_prob','prediction':f'{m}_pred','fold':f'{m}_fold'})
        merged = merged.merge(p, on='window_idx', how='left')
    merged.to_csv(f'{folder}/{ds}_all_predictions.csv', index=False)
    print(f'  ✓ {ds}_all_predictions.csv  ({merged.shape[1]} cols, '
          f'{os.path.getsize(f"{folder}/{ds}_all_predictions.csv")/1e6:.1f} MB)')
    del merged, base; gc.collect()

    # 2. efficiency_all.csv
    rows = []
    for m in EXPECTED:
        f = f'{folder}/{m}_efficiency.csv'
        if not os.path.exists(f): continue
        r = pd.read_csv(f).iloc[0].to_dict(); r['model'] = m
        res = f'{folder}/{m}_results.csv'
        if os.path.exists(res):
            rr = pd.read_csv(res).iloc[0]
            r['auc'] = rr['auc']; r['f1'] = rr['f1']
        rows.append(r)
    df_eff = pd.DataFrame(rows)
    if 'auc' in df_eff.columns:
        df_eff = df_eff.sort_values('auc', ascending=False)
    df_eff.to_csv(f'{folder}/{ds}_efficiency_all.csv', index=False)
    print(f'  ✓ {ds}_efficiency_all.csv  ({len(df_eff)} models)')

    # 3. tsne_coords.csv
    np.random.seed(42)
    nA = min(N_PER_CLASS, (lab==1).sum())
    nN = min(N_PER_CLASS, (lab==0).sum())
    idx_af  = np.random.choice(np.where(lab==1)[0], nA, replace=False)
    idx_nrm = np.random.choice(np.where(lab==0)[0], nN, replace=False)
    sub_idx = np.concatenate([idx_af, idx_nrm]); np.random.shuffle(sub_idx)
    sub_lbl = lab[sub_idx]
    tsne_df = pd.DataFrame({'window_idx':sub_idx,
                            'record_name':recs[sub_idx],
                            'true_label':sub_lbl})
    for m in EXPECTED:
        npy = f'{folder}/{m}_embs.npy'
        if not os.path.exists(npy): continue
        print(f'    t-SNE {m}...', end=' ', flush=True)
        e = np.load(npy, mmap_mode='r')
        sub_emb = StandardScaler().fit_transform(np.array(e[sub_idx]))
        proj = TSNE(n_components=2, perplexity=40, max_iter=1000,
                    random_state=42, n_jobs=-1).fit_transform(sub_emb)
        tsne_df[f'{m}_x'] = proj[:,0]
        tsne_df[f'{m}_y'] = proj[:,1]
        del e, sub_emb, proj; gc.collect()
        print('✓')
    tsne_df.to_csv(f'{folder}/{ds}_tsne_coords.csv', index=False)
    print(f'  ✓ {ds}_tsne_coords.csv  ({tsne_df.shape[1]} cols)')

print('\n🎉 ALL 12 FILES GENERATED (3 per dataset × 4 datasets)')


  CinC2017
  ✓ CinC2017_all_predictions.csv  (30 cols, 7.1 MB)
  ✓ CinC2017_efficiency_all.csv  (9 models)
    t-SNE HuBERT-Small... ✓
    t-SNE HuBERT-Base... ✓
    t-SNE HuBERT-Large... ✓
    t-SNE CLEF-Small... ✓
    t-SNE CLEF-Medium... ✓
    t-SNE CLEF-Large... ✓
    t-SNE ST-MEM... ✓
    t-SNE ECG-JEPA... ✓
    t-SNE ECGFounder... ✓
  ✓ CinC2017_tsne_coords.csv  (21 cols)

  AFDB
  ✓ AFDB_all_predictions.csv  (30 cols, 37.6 MB)
  ✓ AFDB_efficiency_all.csv  (9 models)
    t-SNE HuBERT-Small... ✓
    t-SNE HuBERT-Base... ✓
    t-SNE HuBERT-Large... ✓
    t-SNE CLEF-Small... ✓
    t-SNE CLEF-Medium... ✓
    t-SNE CLEF-Large... ✓
    t-SNE ST-MEM... ✓
    t-SNE ECG-JEPA... ✓
    t-SNE ECGFounder... ✓
  ✓ AFDB_tsne_coords.csv  (21 cols)

  LTAFDB
  ✓ LTAFDB_all_predictions.csv  (30 cols, 308.8 MB)
  ✓ LTAFDB_efficiency_all.csv  (9 models)
    t-SNE HuBERT-Small... ✓
    t-SNE HuBERT-Base... ✓
    t-SNE HuBERT-Large... ✓
    t-SNE CLEF-Small... ✓
    t-SNE CLEF-Medium... ✓
    t-SNE C

In [7]:
import os, zipfile

BASE = '/content/drive/MyDrive/ECG project Amirhossein taleshi'
FOLDERS = {
    'CinC2017': f'{BASE}/results/FILES of BENCHMARK/CinC2017_benchmark',
    'AFDB'    : f'{BASE}/results/FILES of BENCHMARK/AFDB_benchmark',
    'LTAFDB'  : f'{BASE}/results/FILES of BENCHMARK/LTAFDB_benchmark(NEW AND CORRECT)',
    'CPSC2021': f'{BASE}/results/FILES of BENCHMARK/CPSC2021_benchmark',
}

OUT = f'{BASE}/results/PROFESSOR_DELIVERABLES'
os.makedirs(OUT, exist_ok=True)

for ds, folder in FOLDERS.items():
    zip_path = f'{OUT}/{ds}_FINAL.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(os.listdir(folder)):
            if f.endswith(('.csv','.png')) and not f.startswith('_'):
                zf.write(os.path.join(folder, f), f)
    print(f'✓ {ds}_FINAL.zip  ({os.path.getsize(zip_path)/1e6:.1f} MB)')

print(f'\nAll 4 ZIPs in: {OUT}')

✓ CinC2017_FINAL.zip  (7.4 MB)
✓ AFDB_FINAL.zip  (33.1 MB)
✓ LTAFDB_FINAL.zip  (262.2 MB)
✓ CPSC2021_FINAL.zip  (65.7 MB)

All 4 ZIPs in: /content/drive/MyDrive/ECG project Amirhossein taleshi/results/PROFESSOR_DELIVERABLES
